In [ ]:
import os
import numpy as np
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.preprocessing import image
from tensorflow.keras.utils import to_categorical
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from keras.preprocessing.sequence import pad_sequences
import tensorflow as tf
from tensorflow.keras import backend as K

# Set GPU memory growth
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

def extract_frame_number(frame_path):
    base_name = os.path.basename(frame_path)
    frame_number = ''.join(filter(str.isdigit, base_name))
    return int(frame_number)

def build_cnn():
    base_model = VGG16(weights='imagenet', include_top=False)
    model = Model(inputs=base_model.input, outputs=base_model.get_layer('block5_pool').output)
    return model

def preprocess_image(frame_path):
    img = image.load_img(frame_path, target_size=(224, 224))
    img_data = image.img_to_array(img)
    img_data = np.expand_dims(img_data, axis=0)
    img_data = preprocess_input(img_data)
    return img_data

def extract_features_batch(frame_paths, cnn_model, progress_bar):
    with ThreadPoolExecutor() as executor:
        img_data_list = list(tqdm(executor.map(preprocess_image, frame_paths), total=len(frame_paths), desc="Preprocessing images", leave=False))
    img_data_batch = np.vstack(img_data_list)
    features_batch = cnn_model.predict(img_data_batch)
    progress_bar.update(len(frame_paths))  # Update the progress bar by the number of images processed
    return features_batch

def extract_features(frame_paths, cnn_model, progress_bar):
    batch_size = 8  # Adjust batch size as needed
    features = []
    for i in tqdm(range(0, len(frame_paths), batch_size), desc="Extracting features", leave=False):
        batch_paths = frame_paths[i:i + batch_size]
        features_batch = extract_features_batch(batch_paths, cnn_model, progress_bar)
        features.extend(features_batch)
    return np.array(features).reshape(len(frame_paths), 7*7*512)  # Reshape to (num_samples, features)

def load_data(video_dirs, cnn_model):
    X, y = [], []
    total_images = sum(len([f for f in os.listdir(video_dir) if f.endswith('.jpg')]) for video_dir, _ in video_dirs)
    with tqdm(total=total_images, desc="Processing images") as progress_bar:
        for video_dir, label in tqdm(video_dirs, desc="Processing videos"):
            frame_paths = [os.path.join(video_dir, f) for f in os.listdir(video_dir) if f.endswith('.jpg')]
            frame_paths.sort(key=extract_frame_number)
            features = extract_features(frame_paths, cnn_model, progress_bar)
            X.append(features)
            y.append(label)
    # Pad sequences to have the same length
    X = pad_sequences(X, padding='post', dtype='float32')
    return np.array(X), np.array(y)

train_video_dirs = [
    ('vdos/weapon/finalimages/video_0', 0),
    ('vdos/weapon/finalimages/video_0 (1)', 0),
    ('vdos/weapon/finalimages/video_1 (1)', 0),
    ('vdos/weapon/finalimages/video_1', 0),
    ('vdos/weapon/finalimages/video_2 (1)', 0),
    ('vdos/weapon/finalimages/video_2', 0),
    ('vdos/weapon/finalimages/video_3 (1)', 0),
    # ('vdos/weapon/finalimages/video_3', 0),
    # ('vdos/weapon/finalimages/video_4 (1)', 0),
    # ('vdos/weapon/finalimages/video_4', 0),

    ('vdos/safe/finalimages/video_0 (1)', 1),
    ('vdos/safe/finalimages/video_0', 1),
    ('vdos/safe/finalimages/video_1', 1),
    ('vdos/safe/finalimages/video_1 (1)', 1),
    ('vdos/safe/finalimages/video_2', 1),
    ('vdos/safe/finalimages/video_2 (1)', 1),
    ('vdos/safe/finalimages/video_3', 1),
    # ('vdos/safe/finalimages/video_3 (1)', 1),
    # ('vdos/safe/finalimages/video_4', 1),
    # ('vdos/safe/finalimages/video_4 (1)', 1),

    ('vdos/action/finalimages/trimmed_video_0', 3),
    ('vdos/action/finalimages/trimmed_video_0 (1)', 3),
    ('vdos/action/finalimages/trimmed_video_1', 3),
    ('vdos/action/finalimages/trimmed_video_1 (1)', 3),
    ('vdos/action/finalimages/trimmed_video_2', 3),
    ('vdos/action/finalimages/trimmed_video_2 (1)', 3),
    ('vdos/action/finalimages/trimmed_video_3', 3),
    # ('vdos/action/finalimages/trimmed_video_3 (1)', 3),
    # ('vdos/action/finalimages/trimmed_video_4 (1)', 3),
    # ('vdos/action/finalimages/trimmed_video_4', 3),

    ('vdos/fight/finalimages/trimmed_video_0', 2),
    ('vdos/fight/finalimages/trimmed_video_0 (1)', 2),
    ('vdos/fight/finalimages/trimmed_video_1', 2),
    ('vdos/fight/finalimages/trimmed_video_1 (1)', 2),
    ('vdos/fight/finalimages/trimmed_video_2', 2),
    ('vdos/fight/finalimages/trimmed_video_2 (1)', 2),
    ('vdos/fight/finalimages/trimmed_video_3', 2),
    # ('vdos/fight/finalimages/trimmed_video_3 (1)', 2),
    # ('vdos/fight/finalimages/trimmed_video_4 (1)', 2),
    # ('vdos/fight/finalimages/trimmed_video_4', 2),

    ('vdos/blood/finalimages/trimmed_video_0', 4),
    ('vdos/blood/finalimages/trimmed_video_0 (1)', 4),
    ('vdos/blood/finalimages/trimmed_video_1', 4),
    ('vdos/blood/finalimages/trimmed_video_1 (1)', 4),
    ('vdos/blood/finalimages/trimmed_video_2', 4),
    ('vdos/blood/finalimages/trimmed_video_2 (1)', 4),
    ('vdos/blood/finalimages/trimmed_video_3', 4),
    # ('vdos/blood/finalimages/trimmed_video_3 (1)', 4),
    # ('vdos/blood/finalimages/trimmed_video_4 (1)', 4),
    # ('vdos/blood/finalimages/trimmed_video_4', 4)
]

val_video_dirs = [
    ('vdos/safe/video_7', 1),
    ('vdos/safe/video_8', 1),
    ('vdos/safe/video_9', 1),
    ('vdos/weapon/video_5', 0),
    ('vdos/weapon/video_5 (1)', 0),
    ('vdos/weapon/video_4 (1)', 0),
    ('vdos/weapon/video_4', 0),
    ('vdos/fight/trimmed_video_5', 2),
    ('vdos/action/trimmed_video_5', 3),
    ('vdos/blood/trimmed_video_5', 4),
    ('vdos/action/trimmed_video_5 (1)', 3)
]

cnn_model = build_cnn()
X_train, y_train = load_data(train_video_dirs, cnn_model)
X_val, y_val = load_data(val_video_dirs, cnn_model)

# Define LSTM model
lstm_model = Sequential([
    LSTM(256, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=False),
    Dropout(0.5),
    Dense(5, activation='softmax')
])

lstm_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
lstm_model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10, batch_size=8)

# Save the model
lstm_model.save('videoclassification.h5')

print("Model training complete and saved as 'videoclassification.h5'")

# Classify new video
def classify_video(video_path, cnn_model, lstm_model):
    frame_paths = [os.path.join(video_path, f) for f in os.listdir(video_path) if f.endswith('.jpg')]
    frame_paths.sort(key=extract_frame_number)
    features = extract_features(frame_paths, cnn_model)
    features = np.expand_dims(features, axis=0)  # Add batch dimension
    prediction = lstm_model.predict(features)
    predicted_class = np.argmax(prediction, axis=1)[0]
    return class_names[predicted_class]

class_names = ['Weapon', 'Safe', 'Fight', 'Action', 'Blood']

new_video_path = 'vdos/weapon/video_5 (1)'
predicted_category = classify_video(new_video_path, cnn_model, lstm_model)
print(f'The video belongs to the category: {predicted_category}')
